# ML Capstone — Classification Part B
## Member 3: Tharun | Random Forest, AdaBoost, Gradient Boosting

This is a **standalone notebook** for the first three Classification Part B algorithms required for Review 2.

The Review 1 notebook is not embedded or appended here. This notebook independently loads the dataset, checks the target, preprocesses the features, creates a stratified train/test split, trains the three classifiers, tunes their hyperparameters, and compares their test-set metrics.

**Dataset used in the Review 1 notebook:** `global_air_pollution_data.csv`  
Place the CSV in the same folder as this notebook, or update `DATA_PATH` below.

**Reproducibility:** random state = 42. Preprocessing is fitted only on the training split to avoid data leakage.

In [ ]:
# 1. Import libraries
import copy
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, ConfusionMatrixDisplay, classification_report
)

RANDOM_STATE = 42
DATA_PATH = "global_air_pollution_data.csv"

print("Libraries imported.")

## 2. Load and inspect the dataset

The target is `aqi_category`, matching the Review 1 notebook. The same leakage-prone columns, identifier column, and engineered feature approach are used as in Review 1 so the Part A and Part B results are more comparable.

In [ ]:
# Load the dataset
df = pd.read_csv(DATA_PATH)

TARGET_COLUMN = "aqi_category"

print("Dataset shape:", df.shape)
display(df.head())

if TARGET_COLUMN not in df.columns:
    raise ValueError(
        f"Target column '{TARGET_COLUMN}' was not found. "
        f"Available columns: {df.columns.tolist()}"
    )

print("\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False).to_frame("missing_count"))

print("\nTarget distribution:")
display(df[TARGET_COLUMN].value_counts(dropna=False).to_frame("count"))

## 3. Prepare features and target consistently with Review 1

The following columns are removed because they can reveal the target directly or mainly identify a location:
- `aqi_value`
- `co_aqi_category`
- `ozone_aqi_category`
- `no2_aqi_category`
- `pm2.5_aqi_category`
- `city_name`

The engineered feature `average_pollutant_aqi` is retained if it was created in the dataset by the Review 1 workflow. If the raw CSV does not already contain it, the code below creates it from available pollutant AQI columns. Missing target rows are removed. Feature missing values are handled later inside the preprocessing pipeline.

In [ ]:
# Remove target-leakage and identifier columns, matching Review 1.
leakage_columns = [
    "aqi_value",
    "co_aqi_category",
    "ozone_aqi_category",
    "no2_aqi_category",
    "pm2.5_aqi_category"
]
identifier_columns = ["city_name"]

columns_to_drop = [
    col for col in leakage_columns + identifier_columns
    if col in df.columns
]

classification_df = df.drop(columns=columns_to_drop).dropna(
    subset=[TARGET_COLUMN]
).copy()

# Match the Review 1 engineered feature where raw pollutant AQI values exist.
# Do not overwrite it if the feature has already been created.
pollutant_aqi_columns = [
    col for col in ["co_aqi_value", "ozone_aqi_value", "no2_aqi_value", "pm2.5_aqi_value"]
    if col in classification_df.columns
]
if "average_pollutant_aqi" not in classification_df.columns and pollutant_aqi_columns:
    classification_df["average_pollutant_aqi"] = (
        classification_df[pollutant_aqi_columns].mean(axis=1)
    )

X = classification_df.drop(columns=[TARGET_COLUMN])
y = classification_df[TARGET_COLUMN].astype(str)

# Remove only completely empty feature columns.
all_missing_columns = X.columns[X.isna().all()].tolist()
if all_missing_columns:
    print("Dropping entirely missing feature columns:", all_missing_columns)
    X = X.drop(columns=all_missing_columns)

print("Target:", TARGET_COLUMN)
print("Rows:", len(X))
print("Feature count:", X.shape[1])
print("Dropped leakage/identifier columns:", columns_to_drop)
print("Target classes:", sorted(y.unique()))
display(y.value_counts().to_frame("class_count"))

## 4. Stratified train/test split and preprocessing

This uses the same 80:20 stratified split and `random_state=42` convention as Review 1. Numerical features use median imputation followed by `StandardScaler`; categorical features use most-frequent imputation followed by One-Hot Encoding. The preprocessing is inside each model's pipeline, so it is fitted only on the relevant training fold.

In [ ]:
# Create the same stratified 80:20 split convention as Review 1.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Compatible with newer and older scikit-learn versions.
try:
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse=False)

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", one_hot)
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Numerical preprocessing: median imputation -> StandardScaler")
print("Categorical preprocessing: most-frequent imputation -> One-Hot Encoding")

## 5. Evaluation function

This function predicts the test labels, calculates the required metrics, prints a classification report, and displays a confusion matrix and metrics table for each model. Weighted Precision, Recall, and F1 are used so each class contributes according to its size. ROC-AUC uses One-vs-Rest for multiclass data.

In [ ]:
def evaluate_model(model_name, fitted_pipeline):
    # Predict the classes for the test data.
    y_pred = fitted_pipeline.predict(X_test)

    # Get class probabilities for ROC-AUC.
    y_prob = fitted_pipeline.predict_proba(X_test)
    model_classes = fitted_pipeline.named_steps["model"].classes_

    # Calculate ROC-AUC for binary or multiclass classification.
    if len(model_classes) == 2:
        roc_auc = roc_auc_score(y_test, y_prob[:, 1])
    else:
        roc_auc = roc_auc_score(
            y_test, y_prob,
            labels=model_classes,
            multi_class="ovr",
            average="weighted"
        )

    # Calculate all required metrics.
    result = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (weighted)": precision_score(
            y_test, y_pred, average="weighted", zero_division=0
        ),
        "Recall (weighted)": recall_score(
            y_test, y_pred, average="weighted", zero_division=0
        ),
        "F1 (weighted)": f1_score(
            y_test, y_pred, average="weighted", zero_division=0
        ),
        "ROC-AUC (OvR/weighted)": roc_auc
    }

    print("\nClassification report:", model_name)
    print(classification_report(y_test, y_pred, zero_division=0))

    # Show the confusion matrix.
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred,
        labels=model_classes,
        display_labels=model_classes,
        xticks_rotation=45,
        cmap="Blues",
        values_format="d"
    )
    plt.title("Confusion Matrix - " + model_name)
    plt.tight_layout()
    plt.show()

    # Display metrics in a table.
    display(pd.DataFrame([result]).set_index("Model"))
    return result

## 6. Random Forest Classifier

Random Forest trains many decision trees and combines their predictions. We use GridSearchCV to compare a few values for the number of trees and maximum depth. The best setting is selected using 5-fold cross-validation weighted F1.

In [ ]:
# Random Forest: tune the number of trees and maximum tree depth.
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1
    ))
])

rf_parameters = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 15, 25]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rf_search = GridSearchCV(
    rf_pipeline, rf_parameters,
    scoring="f1_weighted", cv=cv, n_jobs=-1, refit=True, verbose=1
)
rf_search.fit(X_train, y_train)

rf_model = rf_search.best_estimator_
print("Best Random Forest parameters:", rf_search.best_params_)
print("Best 5-fold CV weighted F1:", round(rf_search.best_score_, 4))
rf_result = evaluate_model("Random Forest", rf_model)

### Random Forest feature importance

The feature names are obtained from the fitted preprocessing pipeline, so one-hot encoded categories are represented individually. The chart displays the 15 highest-importance transformed features. Treat feature importance as an indication of model reliance, not proof of causation.

In [ ]:
# Get the feature names created by preprocessing and their Random Forest importance.
feature_names = rf_model.named_steps["preprocessor"].get_feature_names_out()
importance = rf_model.named_steps["model"].feature_importances_

importance_table = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance
}).sort_values("Importance", ascending=False)

display(importance_table.head(15))

# Plot the 15 most important features.
top_features = importance_table.head(15).sort_values("Importance")
plt.figure(figsize=(9, 6))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.title("Random Forest - Top 15 Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## 7. AdaBoost Classifier

AdaBoost trains weak models in sequence, giving more attention to examples that were previously classified incorrectly. A decision stump (a tree with depth 1) is used as the base learner. GridSearchCV tunes the number of learners and learning rate.

In [ ]:
# AdaBoost: use a decision stump as the weak learner.
try:
    ada_classifier = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE),
        random_state=RANDOM_STATE
    )
except TypeError:
    # Compatibility with older scikit-learn versions.
    ada_classifier = AdaBoostClassifier(
        base_estimator=DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE),
        random_state=RANDOM_STATE
    )

ada_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", ada_classifier)
])

ada_parameters = {
    "model__n_estimators": [50, 100, 200],
    "model__learning_rate": [0.1, 0.5, 1.0]
}

ada_search = GridSearchCV(
    ada_pipeline, ada_parameters,
    scoring="f1_weighted", cv=cv, n_jobs=-1, refit=True, verbose=1
)
ada_search.fit(X_train, y_train)

ada_model = ada_search.best_estimator_
print("Best AdaBoost parameters:", ada_search.best_params_)
print("Best 5-fold CV weighted F1:", round(ada_search.best_score_, 4))
ada_result = evaluate_model("AdaBoost", ada_model)

## 8. Gradient Boosting Classifier

Gradient Boosting adds trees sequentially, with each new tree helping reduce the ensemble's remaining errors. This section uses scikit-learn's `GradientBoostingClassifier` and tunes `learning_rate`, `n_estimators`, and `max_depth`.

## 8. Gradient Boosting Classifier

Gradient Boosting adds decision trees one after another, with each new tree helping correct remaining errors. GridSearchCV tunes the learning rate, number of trees, and tree depth.

In [ ]:
# Gradient Boosting: tune learning rate, number of trees, and tree depth.
gb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", GradientBoostingClassifier(random_state=RANDOM_STATE))
])

gb_parameters = {
    "model__n_estimators": [100, 200],
    "model__learning_rate": [0.03, 0.1, 0.2],
    "model__max_depth": [2, 3]
}

gb_search = GridSearchCV(
    gb_pipeline, gb_parameters,
    scoring="f1_weighted", cv=cv, n_jobs=-1, refit=True, verbose=1
)
gb_search.fit(X_train, y_train)

gb_model = gb_search.best_estimator_
print("Best Gradient Boosting parameters:", gb_search.best_params_)
print("Best 5-fold CV weighted F1:", round(gb_search.best_score_, 4))
gb_result = evaluate_model("Gradient Boosting", gb_model)

## 9. Comparison table — first three Part B models

This table compares the three models on the same held-out test set. The values appear after all three model sections have run successfully.

In [ ]:
# Compare the three models using the same test set.
results_part_b = pd.DataFrame([
    rf_result,
    ada_result,
    gb_result
]).sort_values("F1 (weighted)", ascending=False).reset_index(drop=True)

display(results_part_b.round(4))

print("Top model among these three by weighted F1:")
display(results_part_b.head(1).round(4))